# Real U50 Fast Uniform Benchmark: GAP + MILP + Genetic

Быстрый ноутбук для уменьшенного датасета, где задачи урезаются пространственно-равномерно,
а агенты урезаются равномерно по депо с запасом по сменному времени.

Основной параметр: `DOWNSCALE_FACTOR` (например, `2..10`).


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import importlib
import inspect
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Clean cached modules to avoid stale code in long-lived kernels.
for mod_name in list(sys.modules):
    if mod_name.startswith("flowopt"):
        sys.modules.pop(mod_name, None)

import flowopt.pipeline as fp
from flowopt.pipelines.u50_fast_dataset import build_fast_u50_dataset

fp = importlib.reload(fp)

run_real_gap_vrp = fp.run_real_gap_vrp
run_real_milp = fp.run_real_milp
run_real_genetic = fp.run_real_genetic

milp_sig = inspect.signature(run_real_milp)
if "time_limit_sec" not in milp_sig.parameters:
    raise RuntimeError(f"Stale run_real_milp loaded: signature={milp_sig}")

pd.set_option("display.max_colwidth", 160)
print("REPO_ROOT:", REPO_ROOT)
print("pipeline module:", fp.__file__)
print("run_real_milp signature:", milp_sig)


REPO_ROOT: C:\Users\Пользователь\Optimization-of-flows-3
pipeline module: C:\Users\Пользователь\Optimization-of-flows-3\src\flowopt\pipeline.py
run_real_milp signature: (*, dataset_path: 'Path | str' = WindowsPath('C:/Users/Пользователь/Optimization-of-flows-3/src/data/real/real_simple/dataset_real_spb_simple.json'), time_limit_sec: 'int' = 60, unassigned_penalty: 'float' = 100000.0, show_progress: 'bool' = False, progress_hook: 'Callable[[str], None] | None' = None) -> 'RunMetrics'


In [7]:
# Build reduced dataset variant (uniform by depot/space)
BASE_DATASET_PATH = REPO_ROOT / "src" / "data" / "real" / "day_load_profiles" / "u50" / "simplified_a_only" / "dataset_real_spb_day_u50_A_only_simplified.json"
FAST_ROOT = REPO_ROOT / "src" / "data" / "real" / "day_load_profiles" / "u50" / "fast_uniform"
FAST_ROOT.mkdir(parents=True, exist_ok=True)

DOWNSCALE_FACTOR = 10  # try 2..10
SEED = 42
GRID_SIZE = 6
SAFETY_UTILIZATION = 0.85
FORCE_REBUILD = False

factor_tag = f"f{int(DOWNSCALE_FACTOR) if float(DOWNSCALE_FACTOR).is_integer() else str(DOWNSCALE_FACTOR).replace('.', '_')}"
DATASET_PATH = FAST_ROOT / f"dataset_real_spb_day_u50_fast_{factor_tag}.json"
SUMMARY_PATH = FAST_ROOT / f"summary_real_spb_day_u50_fast_{factor_tag}.json"

if FORCE_REBUILD or not DATASET_PATH.exists() or not SUMMARY_PATH.exists():
    summary = build_fast_u50_dataset(
        base_dataset_path=BASE_DATASET_PATH,
        out_dataset_path=DATASET_PATH,
        downscale_factor=float(DOWNSCALE_FACTOR),
        seed=SEED,
        grid_size=GRID_SIZE,
        safety_utilization=SAFETY_UTILIZATION,
    )
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))

print("BASE_DATASET_PATH:", BASE_DATASET_PATH)
print("DATASET_PATH     :", DATASET_PATH)
print("SUMMARY_PATH     :", SUMMARY_PATH)
print("before ->", summary["before"])
print("after  ->", summary["after"])

pd.DataFrame(
    {
        "depot_id": list(summary["selected_tasks_by_depot"].keys()),
        "tasks_selected": list(summary["selected_tasks_by_depot"].values()),
        "agents_selected": [summary["selected_agents_by_depot"][d] for d in summary["selected_tasks_by_depot"].keys()],
        "required_agents_time_model": [summary["required_agents_by_depot"][d] for d in summary["selected_tasks_by_depot"].keys()],
        "required_hours_time_model": [summary["required_hours_by_depot"][d] for d in summary["selected_tasks_by_depot"].keys()],
    }
).sort_values("tasks_selected", ascending=False)


BASE_DATASET_PATH: C:\Users\Пользователь\Optimization-of-flows-3\src\data\real\day_load_profiles\u50\simplified_a_only\dataset_real_spb_day_u50_A_only_simplified.json
DATASET_PATH     : C:\Users\Пользователь\Optimization-of-flows-3\src\data\real\day_load_profiles\u50\fast_uniform\dataset_real_spb_day_u50_fast_f10.json
SUMMARY_PATH     : C:\Users\Пользователь\Optimization-of-flows-3\src\data\real\day_load_profiles\u50\fast_uniform\summary_real_spb_day_u50_fast_f10.json
before -> {'agents': 432, 'tasks': 301, 'mno_nodes': 265}
after  -> {'agents': 50, 'tasks': 32, 'mno_nodes': 32, 'task_sources': 32}


,depot_id,tasks_selected,agents_selected,required_agents_time_model,required_hours_time_model
0,1533456031,15,9,9,105.742239
3,230361,10,17,5,51.709893
2,1903257537,6,6,3,33.583799
1,1746883131,1,18,1,6.696456


In [8]:
# Dataset inspection + optional profile patching
import flowopt.core as core
import flowopt.genetic_solver_components as ga

with DATASET_PATH.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

meta = dataset.get("metadata", {})
counts = {
    "nodes": len(dataset.get("graph", {}).get("nodes", [])),
    "edges": len(dataset.get("graph", {}).get("edges", [])),
    "agents": len(dataset.get("agents", [])),
    "tasks": len(dataset.get("tasks", [])),
    "objects": len({t.get("destination_node_id") for t in dataset.get("tasks", [])}),
    "sources": len({t.get("source_node_id") for t in dataset.get("tasks", [])}),
}

print("dataset name:", meta.get("name"))
print("counts:", counts)
print("fast_profile:", meta.get("fast_profile", {}))

APPLY_PROFILE_LIMITS = True
if APPLY_PROFILE_LIMITS:
    profiles = meta.get("vehicle_profiles", {})
    for vt, p in profiles.items():
        km = float(p.get("max_daily_km", core.MAX_DAILY_KM_BY_TYPE.get(vt, 130.0)))
        hh = float(p.get("max_shift_hours", core.MAX_SHIFT_HOURS_BY_TYPE.get(vt, 10.0)))
        sp = float(p.get("avg_speed_kmph", core.AVG_SPEED_KMPH_BY_TYPE.get(vt, 24.0)))
        core.MAX_DAILY_KM_BY_TYPE[vt] = km
        core.MAX_SHIFT_HOURS_BY_TYPE[vt] = hh
        core.AVG_SPEED_KMPH_BY_TYPE[vt] = sp
        ga.MAX_DAILY_KM_BY_TYPE[vt] = km
        ga.MAX_SHIFT_HOURS_BY_TYPE[vt] = hh
        ga.AVG_SPEED_KMPH_BY_TYPE[vt] = sp

    print("Patched limits from vehicle_profiles for:", sorted(profiles.keys()))

pd.DataFrame([counts])


dataset name: real_spb_u50_fast_f10
counts: {'nodes': 46730, 'edges': 127100, 'agents': 50, 'tasks': 32, 'objects': 5, 'sources': 32}
fast_profile: {'source_dataset': 'C:\\Users\\Пользователь\\Optimization-of-flows-3\\src\\data\\real\\day_load_profiles\\u50\\simplified_a_only\\dataset_real_spb_day_u50_A_only_simplified.json', 'downscale_factor': 10.0, 'seed': 42, 'grid_size': 6, 'safety_utilization': 0.85, 'selected_tasks': 32, 'selected_agents': 50, 'demoted_mno_to_road': 233, 'required_agents_by_depot': {'1533456031': 9, '1746883131': 1, '1903257537': 3, '230361': 5}, 'selected_agents_by_depot': {'1533456031': 9, '1746883131': 18, '1903257537': 6, '230361': 17}, 'selected_tasks_by_depot': {'1533456031': 15, '1746883131': 1, '1903257537': 6, '230361': 10}, 'required_hours_by_depot': {'1533456031': 105.742239, '1746883131': 6.696456, '1903257537': 33.583799, '230361': 51.709893}}
Patched limits from vehicle_profiles for: ['VT_A', 'VT_AB', 'VT_ABD', 'VT_AD', 'VT_C', 'VT_CD']


,nodes,edges,agents,tasks,objects,sources
0,46730,127100,50,32,5,32


In [9]:
# Fast run of 3 algorithms
from time import perf_counter
from datetime import datetime

# GAP
GAP_STEP1_METHOD = "lp"
GAP_ITER = 20
MAX_AGENTS = None

# MILP
RMILP_TIME_LIMIT_SEC = 90
RMILP_UNASSIGNED_PENALTY = 1e7

# Genetic (kept moderate for speed)
GA_POPULATION_SIZE = 20
GA_GENERATIONS = 20
GA_ELITE_SIZE = 3
GA_SEED = 42
GA_GENERATION_SCALE = 1.0
GA_MIN_GENERATIONS = 10

SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False
REQUIRE_FULL_ASSIGNMENT = False


def make_progress_logger(enabled: bool):
    if not enabled:
        return None
    t0 = perf_counter()
    def _log(message: str) -> None:
        dt = perf_counter() - t0
        print(f"[+{dt:7.1f}s] {message}", flush=True)
    return _log


progress_log = make_progress_logger(SHOW_ALGO_PROGRESS)


def run_with_live_status(label, fn, **kwargs):
    if progress_log is not None:
        progress_log(f"{label}: START")
    ts = perf_counter()
    out = fn(**kwargs)
    if progress_log is not None:
        progress_log(f"{label}: DONE in {perf_counter() - ts:.1f}s")
    return out


results = [
    run_with_live_status(
        "GAP-VRP",
        run_real_gap_vrp,
        dataset_path=DATASET_PATH,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        max_agents=MAX_AGENTS,
        use_repair=True,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-MILP",
        run_real_milp,
        dataset_path=DATASET_PATH,
        time_limit_sec=RMILP_TIME_LIMIT_SEC,
        unassigned_penalty=RMILP_UNASSIGNED_PENALTY,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-GENETIC",
        run_real_genetic,
        dataset_path=DATASET_PATH,
        population_size=GA_POPULATION_SIZE,
        generations=GA_GENERATIONS,
        elite_size=GA_ELITE_SIZE,
        seed=GA_SEED,
        generation_scale=GA_GENERATION_SCALE,
        min_generations=GA_MIN_GENERATIONS,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
]

benchmark_df = pd.DataFrame([r.as_dict() for r in results])
benchmark_df = benchmark_df.sort_values(
    by=["all_checks_ok", "transport_work_ton_km", "total_km", "runtime_sec"],
    ascending=[False, True, True, True],
    na_position="last",
).reset_index(drop=True)

# Save benchmark artifact
LOCAL_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "real_data" / "u50_fast_uniform" / factor_tag
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULT_JSON_PATH = LOCAL_OUT_DIR / f"benchmark_u50_fast_{factor_tag}_{RUN_TAG}.json"

records = benchmark_df.where(pd.notna(benchmark_df), None).to_dict(orient="records")
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "real_u50_fast_uniform_3algo_benchmark.ipynb",
    "dataset_path": str(DATASET_PATH),
    "config": {
        "downscale_factor": DOWNSCALE_FACTOR,
        "seed": SEED,
        "grid_size": GRID_SIZE,
        "safety_utilization": SAFETY_UTILIZATION,
        "gap_step1_method": GAP_STEP1_METHOD,
        "gap_iter": GAP_ITER,
        "max_agents": MAX_AGENTS,
        "rmilp_time_limit_sec": RMILP_TIME_LIMIT_SEC,
        "rmilp_unassigned_penalty": RMILP_UNASSIGNED_PENALTY,
        "ga_population_size": GA_POPULATION_SIZE,
        "ga_generations": GA_GENERATIONS,
        "ga_elite_size": GA_ELITE_SIZE,
        "ga_seed": GA_SEED,
        "ga_generation_scale": GA_GENERATION_SCALE,
        "ga_min_generations": GA_MIN_GENERATIONS,
    },
    "results": records,
}
RESULT_JSON_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULT_JSON_PATH)

if REQUIRE_FULL_ASSIGNMENT:
    if (benchmark_df["unassigned_tasks"].fillna(10**9) > 0).any():
        raise AssertionError("Some algorithms still have unassigned tasks")

main_cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "total_km",
    "deadhead_km",
    "deadhead_share_pct",
    "total_hours",
    "runtime_sec",
    "solver_error",
]
benchmark_df[[c for c in main_cols if c in benchmark_df.columns]]


[+    0.0s] GAP-VRP: START
[+    0.0s] [real_gap_vrp] load dataset: C:\Users\Пользователь\Optimization-of-flows-3\src\data\real\day_load_profiles\u50\fast_uniform\dataset_real_spb_day_u50_fast_f10.json
[+    1.7s] [real_gap_vrp] dataset loaded (nodes=46730, edges=127100, agents=50, tasks=32)
[+    1.7s] [real_gap_vrp] build nx graph
[+    2.1s] [real_gap_vrp] solver start
[+    2.1s] [real_gap_vrp] [GAP-VRP] start: GAP-Lagrangean + VRP(NN+2opt) [step1=lp]
[+    2.1s] [real_gap_vrp] [GAP-VRP] step1: task generation
[+    2.1s] [real_gap_vrp] [GAP-step1] LP setup: sources=32, sinks=9, variables=288, total_waste_tons=37.617
[+    3.2s] [real_gap_vrp] [GAP-step1] distance matrix: sources=1/32, elapsed=1.2s
[+    4.0s] [real_gap_vrp] [GAP-step1] distance matrix: sources=2/32, elapsed=1.9s
[+    4.7s] [real_gap_vrp] [GAP-step1] distance matrix: sources=3/32, elapsed=2.6s
[+    5.5s] [real_gap_vrp] [GAP-step1] distance matrix: sources=4/32, elapsed=3.4s
[+    6.3s] [real_gap_vrp] [GAP-step1] 

,algorithm,feasible,all_checks_ok,assigned_routes,unassigned_tasks,active_agents,transport_work_ton_km,total_km,deadhead_km,deadhead_share_pct,total_hours,runtime_sec
0,real_gap_vrp,True,True,32,0,6,419.865,932.842,556.686,59.676,45.908,1449.481
1,real_milp,True,True,32,0,23,1941.714,4510.761,2257.362,50.044,190.646,107.140
2,real_genetic,True,True,32,0,32,1941.714,4627.014,2373.615,51.299,195.709,342.158


In [10]:
# Schedule map (graph with route scheduling)
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import plotly.express as px

from flowopt.backend.io import load_dataset
from flowopt.solvers import real_gap_vrp_solver, real_milp_solver, real_genetic_solver
from flowopt import gap_vrp_solver as gap_core
from flowopt.dataset import RoutingDataset
import flowopt.core as core

SCHEDULE_ALGO = "real_gap_vrp"  # real_gap_vrp | real_milp | real_genetic
MAP_INTERACTIVE = True
MAP_SHOW_SOLVER_PROGRESS = False

MAP_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "real_data" / "u50_fast_uniform" / factor_tag
MAP_OUT_DIR.mkdir(parents=True, exist_ok=True)
SCHEDULE_MAP_PATH = MAP_OUT_DIR / f"schedule_map_{SCHEDULE_ALGO}_{factor_tag}.png"
SCHEDULE_PLAN_PATH = MAP_OUT_DIR / f"schedule_plan_{SCHEDULE_ALGO}_{factor_tag}.json"

# Re-run selected solver to get full routes/states object for rendering.
dataset_obj, payload_obj = load_dataset(DATASET_PATH)
graph_obj = core.build_nx_graph(dataset_obj)
cache_obj: dict[tuple[str, str], tuple[list[str], float] | None] = {}

if SCHEDULE_ALGO == "real_gap_vrp":
    solver_result = real_gap_vrp_solver.solve_real_gap_vrp(
        dataset=dataset_obj,
        payload=payload_obj,
        graph=graph_obj,
        cache=cache_obj,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        use_repair=True,
        show_progress=MAP_SHOW_SOLVER_PROGRESS,
        verbose=False,
    )
elif SCHEDULE_ALGO == "real_milp":
    solver_result = real_milp_solver.solve_real_milp(
        dataset=dataset_obj,
        payload=payload_obj,
        graph=graph_obj,
        cache=cache_obj,
        time_limit_sec=RMILP_TIME_LIMIT_SEC,
        unassigned_penalty=RMILP_UNASSIGNED_PENALTY,
        show_progress=MAP_SHOW_SOLVER_PROGRESS,
    )
elif SCHEDULE_ALGO == "real_genetic":
    effective_generations = GA_GENERATIONS
    if GA_GENERATION_SCALE > 0:
        effective_generations = max(GA_MIN_GENERATIONS, int(round(GA_GENERATIONS * GA_GENERATION_SCALE)))
    solver_result = real_genetic_solver.solve_real_genetic(
        dataset=dataset_obj,
        payload=payload_obj,
        graph=graph_obj,
        cache=cache_obj,
        population_size=GA_POPULATION_SIZE,
        generations=effective_generations,
        elite_size=GA_ELITE_SIZE,
        seed=GA_SEED,
        show_progress=MAP_SHOW_SOLVER_PROGRESS,
    )
else:
    raise ValueError(f"Unknown SCHEDULE_ALGO={SCHEDULE_ALGO}")

print(
    {
        "algorithm": SCHEDULE_ALGO,
        "assigned_routes": len(solver_result.routes),
        "unassigned": len(solver_result.unassigned),
        "active_agents": sum(1 for s in solver_result.states.values() if s.task_ids),
        "feasible": solver_result.feasible,
    }
)

if not solver_result.routes:
    raise RuntimeError("No routes in solver_result, schedule map cannot be rendered")

# GAP-LP uses generated task ids (LP_T...), so for rendering switch task table to step1-generated tasks.
dataset_for_render = dataset_obj
route_task_ids = {tid for route in solver_result.routes for tid in route.task_ids}
base_task_ids = {t.task_id for t in dataset_obj.tasks}

if not route_task_ids.issubset(base_task_ids):
    if SCHEDULE_ALGO != "real_gap_vrp":
        missing = sorted(route_task_ids - base_task_ids)[:5]
        raise RuntimeError(f"Route task ids missing in dataset tasks: sample={missing}")

    if GAP_STEP1_METHOD == "lp":
        generated_tasks = gap_core.generate_tasks_lp(
            dataset_obj,
            graph_obj,
            cache_obj,
            show_progress=MAP_SHOW_SOLVER_PROGRESS,
        )
    else:
        generated_tasks = gap_core.generate_tasks_greedy(
            dataset_obj,
            graph_obj,
            cache_obj,
            show_progress=MAP_SHOW_SOLVER_PROGRESS,
        )

    generated_task_ids = {t.task_id for t in generated_tasks}
    if not route_task_ids.issubset(generated_task_ids):
        missing = sorted(route_task_ids - generated_task_ids)[:5]
        raise RuntimeError(f"Route task ids missing in generated step1 tasks: sample={missing}")

    dataset_for_render = RoutingDataset(
        graph=dataset_obj.graph,
        fleet=dataset_obj.fleet,
        tasks=generated_tasks,
        routes=dataset_obj.routes,
        metadata=dataset_obj.metadata,
    )
    print(
        "Render uses generated step1 tasks:",
        {"generated_tasks": len(generated_tasks), "method": GAP_STEP1_METHOD},
    )

core.render_solution_map(
    dataset_for_render,
    graph_obj,
    cache_obj,
    solver_result.routes,
    solver_result.states,
    SCHEDULE_MAP_PATH,
)

# Save route plan for further debug/replay.
plan_payload = core.build_solution_plan(
    solver_result.routes,
    dataset_for_render,
    solver_result.states,
)
SCHEDULE_PLAN_PATH.write_text(json.dumps(plan_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved map :", SCHEDULE_MAP_PATH)
print("Saved plan:", SCHEDULE_PLAN_PATH)

img = mpimg.imread(SCHEDULE_MAP_PATH)
if MAP_INTERACTIVE:
    fig = px.imshow(img, title=f"Schedule map: {SCHEDULE_ALGO} ({factor_tag})", origin="upper")
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_layout(width=1200, height=850, margin=dict(l=10, r=10, t=40, b=10))
    fig.show()
else:
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Schedule map: {SCHEDULE_ALGO} ({factor_tag})")
    plt.show()


{'algorithm': 'real_gap_vrp', 'assigned_routes': 32, 'unassigned': 0, 'active_agents': 6, 'feasible': True}
Render uses generated step1 tasks: {'generated_tasks': 32, 'method': 'lp'}
Saved map : C:\Users\Пользователь\Optimization-of-flows-3\notebooks\local\real_data\u50_fast_uniform\f10\schedule_map_real_gap_vrp_f10.png
Saved plan: C:\Users\Пользователь\Optimization-of-flows-3\notebooks\local\real_data\u50_fast_uniform\f10\schedule_plan_real_gap_vrp_f10.json


In [ ]:
# Optional detailed columns
detail_cols = [
    "algorithm",
    "checks",
    "solver_error",
    "step1_method",
    "gap_iter",
    "used_agents",
    "time_limit_sec",
    "unassigned_penalty",
    "population_size",
    "generations",
    "generations_requested",
    "generation_scale",
    "elite_size",
]
benchmark_df[[c for c in detail_cols if c in benchmark_df.columns]]


,algorithm,checks,step1_method,gap_iter,used_agents,time_limit_sec,unassigned_penalty,population_size,generations,generations_requested,generation_scale,elite_size
0,real_gap_vrp,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",lp,20.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,real_milp,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",NaN,NaN,NaN,90.0,10000000.0,NaN,NaN,NaN,NaN,NaN
2,real_genetic,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",NaN,NaN,NaN,NaN,NaN,20.0,20.0,20.0,1.0,3.0


In [ ]:
# Optional: agent/task breakdown tables
from flowopt.reporting import solution_breakdown_tables
from IPython.display import Markdown, display

MAX_AGENTS_TO_SHOW = 30
MAX_TASK_IDS_IN_CELL = 12
MAX_TASK_ROWS_TO_SHOW = 300

for _, row in benchmark_df.iterrows():
    algo = row.get("algorithm", "unknown")
    tables = solution_breakdown_tables(
        row,
        max_agents=MAX_AGENTS_TO_SHOW,
        max_task_ids=MAX_TASK_IDS_IN_CELL,
    )

    display(Markdown(f"### {algo}: решение по агентам"))
    display(tables["summary"])

    if tables["agents"].empty:
        print("No active agents in solution")
        continue

    display(tables["agents"])
    display(Markdown(f"**{algo}: задача -> агент (первые {MAX_TASK_ROWS_TO_SHOW} строк)**"))
    display(tables["tasks"].head(MAX_TASK_ROWS_TO_SHOW))


### real_gap_vrp: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_gap_vrp,6,32,0.0,932.843,45.908


,agent_id,vehicle_type,is_compact,tasks_count,routes_count,total_mass_tons,total_km,deadhead_km,deadhead_share_pct,total_hours,task_ids
0,AGENT_00254,VT_AB,False,10,10,0,237.760,140.055,58.906,12.107,"LP_T0002, LP_T0020, LP_T0010, LP_T0005, LP_T0015, LP_T0012, LP_T0013, LP_T0000, LP_T0021, LP_T0030"
1,AGENT_00316,VT_AB,False,7,7,0,177.783,99.710,56.085,8.948,"LP_T0014, LP_T0009, LP_T0011, LP_T0018, LP_T0006, LP_T0007, LP_T0008"
2,AGENT_00124,VT_AB,False,6,6,0,132.783,66.903,50.385,6.853,"LP_T0023, LP_T0019, LP_T0025, LP_T0001, LP_T0004, LP_T0003"
3,AGENT_00252,VT_AB,False,5,5,0,180.895,107.573,59.467,8.637,"LP_T0016, LP_T0026, LP_T0027, LP_T0028, LP_T0029"
4,AGENT_00251,VT_AB,False,3,3,0,80.262,51.837,64.585,4.004,"LP_T0022, LP_T0024, LP_T0017"
5,AGENT_00516,VT_ABD,False,1,1,0,123.360,90.608,73.450,5.360,LP_T0031


**real_gap_vrp: задача -> агент (первые 300 строк)**

,algorithm,agent_id,task_order,task_id
0,real_gap_vrp,AGENT_00254,1,LP_T0002
1,real_gap_vrp,AGENT_00254,2,LP_T0020
2,real_gap_vrp,AGENT_00254,3,LP_T0010
3,real_gap_vrp,AGENT_00254,4,LP_T0005
4,real_gap_vrp,AGENT_00254,5,LP_T0015
5,real_gap_vrp,AGENT_00254,6,LP_T0012
6,real_gap_vrp,AGENT_00254,7,LP_T0013
7,real_gap_vrp,AGENT_00254,8,LP_T0000
8,real_gap_vrp,AGENT_00254,9,LP_T0021
9,real_gap_vrp,AGENT_00254,10,LP_T0030


...

### real_genetic: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_genetic,32,32,37.617,4627.014,195.709


,agent_id,vehicle_type,is_compact,tasks_count,routes_count,total_mass_tons,total_km,deadhead_km,deadhead_share_pct,total_hours,task_ids
0,AGENT_00083,VT_AB,False,1,1,9.779,32.007,21.488,67.136,1.554,TASK_18897
1,AGENT_00344,VT_AB,False,1,1,3.452,41.332,25.991,62.882,1.942,TASK_25035
2,AGENT_00617,VT_AD,False,1,1,1.088,123.787,62.886,50.802,4.981,TASK_04529
3,AGENT_00020,VT_A,False,1,1,1.063,127.622,65.117,51.024,5.129,TASK_04481
4,AGENT_00333,VT_AB,False,1,1,1.151,128.647,67.503,52.471,5.580,TASK_19520
5,AGENT_00061,VT_AB,False,1,1,1.068,131.162,65.787,50.157,5.685,TASK_02580
6,AGENT_00620,VT_AD,False,1,1,0.937,131.366,66.650,50.736,5.273,TASK_02641
7,AGENT_00316,VT_AB,False,1,1,0.815,131.915,71.766,54.403,5.716,TASK_07218
8,AGENT_00124,VT_AB,False,1,1,1.068,133.778,67.095,50.154,5.794,TASK_02354
9,AGENT_00085,VT_AB,False,1,1,0.933,134.221,67.317,50.154,5.813,TASK_02958


**real_genetic: задача -> агент (первые 300 строк)**

,algorithm,agent_id,task_order,task_id
0,real_genetic,AGENT_00083,1,TASK_18897
1,real_genetic,AGENT_00344,1,TASK_25035
2,real_genetic,AGENT_00617,1,TASK_04529
3,real_genetic,AGENT_00020,1,TASK_04481
4,real_genetic,AGENT_00333,1,TASK_19520
5,real_genetic,AGENT_00061,1,TASK_02580
6,real_genetic,AGENT_00620,1,TASK_02641
7,real_genetic,AGENT_00316,1,TASK_07218
8,real_genetic,AGENT_00124,1,TASK_02354
9,real_genetic,AGENT_00085,1,TASK_02958
